# 🎨 Module 2: Stable Diffusion 1.5 Cartoon LoRA Training
### Automated, Memory-Optimized Training Engine for Google Colab & Kaggle (Nvidia T4 GPU)

> **Part of the Image-to-Cartoon-Video Pipeline**  
> Trains a parameter-efficient LoRA adapter on UNet attention layers (`to_k`, `to_q`, `to_v`, `to_out.0`) targeting Stable Diffusion 1.5.

---
### ⚡ Hardware Prerequisites
1. In Google Colab: navigate to **Runtime** > **Change runtime type** > select **T4 GPU** (or any GPU runtime).
2. Ensure you have ~10 GB free disk space.

## 1. Verify GPU Allocation & VRAM

In [ ]:
!nvidia-smi

## 2. Install Required Dependencies
Installs PyTorch, Hugging Face `diffusers`, `peft`, `accelerate`, and the `bitsandbytes` 8-bit optimizer.

In [ ]:
# Install optimized packages for T4 training
!pip install -q --upgrade pip
!pip install -q \
    diffusers==0.27.2 \
    transformers==4.38.2 \
    peft==0.9.0 \
    accelerate==0.28.0 \
    bitsandbytes>=0.43.0 \
    safetensors>=0.4.2 \
    torchvision \
    tqdm \
    scipy

## 3. Mount Google Drive (Optional) or Prepare Dataset
If your curated dataset from **Module 1** is in Google Drive, mount it below, or extract a dataset zip file.

In [ ]:
# Optional: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted successfully.")
except Exception:
    print("Not running in Google Colab or Drive mount skipped.")

# Set dataset and output paths
import os
DATASET_DIR = "/content/lora_dataset"
OUTPUT_DIR = "/content/cartoon_lora_sd15_output"

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# If you have a zip file of your dataset from Module 1, unzip it:
# !unzip -q /content/drive/MyDrive/lora_dataset.zip -d /content/lora_dataset

## 4. Configure Accelerate for Single-GPU Mixed Precision (`fp16`)

In [ ]:
import json
import os

# Write standard single-GPU accelerate config
accel_config = {
    "compute_environment": "LOCAL_MACHINE",
    "distributed_type": "NO",
    "mixed_precision": "fp16",
    "num_machines": 1,
    "num_processes": 1,
    "use_cpu": False,
}

os.makedirs("/root/.cache/huggingface/accelerate", exist_ok=True)
with open("/root/.cache/huggingface/accelerate/default_config.yaml", "w") as f:
    import yaml
    yaml.dump(accel_config, f)

print("Accelerate configuration saved for fp16 training.")

## 5. Launch LoRA Training Engine

The training engine uses:
- **8-bit AdamW** optimizer (`bitsandbytes`) to fit in < 10 GB VRAM.
- **Gradient checkpointing** on the UNet attention layers.
- **Gradient accumulation steps = 4** (Effective batch size = 4 with per-device batch size = 1).
- **Validation image generation** every 250 steps to track style convergence in real-time.

In [ ]:
!accelerate launch train_lora.py \
    --dataset_dir="/content/lora_dataset" \
    --output_dir="/content/cartoon_lora_sd15_output" \
    --pretrained_model_name_or_path="runwayml/stable-diffusion-v1-5" \
    --trigger_word="in cartoon artstyle," \
    --lora_rank=16 \
    --lora_alpha=16 \
    --train_batch_size=1 \
    --gradient_accumulation_steps=4 \
    --learning_rate=1e-4 \
    --lr_scheduler="cosine" \
    --lr_warmup_steps=100 \
    --max_train_steps=1500 \
    --mixed_precision="fp16" \
    --use_8bit_adam \
    --gradient_checkpointing \
    --checkpointing_steps=500 \
    --validation_steps=250 \
    --validation_prompt="in cartoon artstyle, a brave warrior with a glowing sword standing on a hill" \
    --num_validation_images=2 \
    --report_to="tensorboard"

## 6. Inspect Real-Time Validation Samples
Display intermediate samples generated during training to inspect how the cartoon style converged across checkpoints.

In [ ]:
import glob
import matplotlib.pyplot as plt
from PIL import Image

val_images = sorted(glob.glob("/content/cartoon_lora_sd15_output/validation_samples/*.png"))
print(f"Found {len(val_images)} validation images.")

if val_images:
    cols = min(4, len(val_images))
    rows = (len(val_images) + cols - 1) // cols
    plt.figure(figsize=(5 * cols, 5 * rows))
    for idx, img_path in enumerate(val_images[-8:]):  # Display up to last 8 samples
        plt.subplot(rows, cols, idx + 1)
        plt.imshow(Image.open(img_path))
        plt.title(os.path.basename(img_path), fontsize=9)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

## 7. Interactive Test Inference with Your Trained LoRA
Load the trained `.safetensors` LoRA weights into a `StableDiffusionPipeline` and generate high-resolution cartoon images.

In [ ]:
import torch
from diffusers import StableDiffusionPipeline

model_id = "runwayml/stable-diffusion-v1-5"
lora_dir = "/content/cartoon_lora_sd15_output"

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    safety_checker=None
).to("cuda")

# Load trained PEFT LoRA weights
pipe.load_lora_weights(lora_dir)
print("Trained LoRA weights loaded into pipeline.")

# Generate sample cartoon images
prompt = "in cartoon artstyle, a futuristic city with flying cars and neon cherry blossom trees, masterwork"
negative_prompt = "photorealistic, realistic, photograph, deformed, ugly, blurry, low quality"

generator = torch.Generator(device="cuda").manual_seed(42)
image = pipe(
    prompt=prompt,
    negative_prompt=negative_prompt,
    num_inference_steps=30,
    guidance_scale=7.5,
    cross_attention_kwargs={"scale": 0.85},  # Adjust LoRA strength (0.6 - 1.0)
    generator=generator
).images[0]

# Display result
image.save("/content/test_output.png")
image

## 8. Export and Download LoRA Weights
Compress and download your standalone `.safetensors` LoRA model for use in ComfyUI, AUTOMATIC1111 WebUI, or downstream video generation.

In [ ]:
# Zip final weights and metadata for download
!zip -r /content/cartoon_lora_sd15.zip /content/cartoon_lora_sd15_output -x "*.bin"

try:
    from google.colab import files
    files.download('/content/cartoon_lora_sd15.zip')
    print("Download initiated.")
except Exception:
    print("File saved at: /content/cartoon_lora_sd15.zip")